# Drug-Target Binding Affinity Predictor
## Demo Notebook

This notebook demonstrates the full pipeline:
1. Load the Davis Kinase dataset
2. Featurize drugs and proteins
3. Train a GNN model
4. Evaluate on test set
5. Make predictions on new drug-target pairs

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 1. Load Dataset

In [ ]:
from src.data_loader import DavisDataset

dataset = DavisDataset(data_dir='data/raw')
df = dataset.load()

print(f"Total pairs: {len(df)}")
print(f"Unique drugs: {df['drug_smiles'].nunique()}")
print(f"Unique proteins: {df['protein_sequence'].nunique()}")
print(f"pKd range: {df['kd_value'].min():.2f} - {df['kd_value'].max():.2f}")
print(f"pKd mean: {df['kd_value'].mean():.2f} +/- {df['kd_value'].std():.2f}")

df.head()

## 2. Featurize Drugs and Proteins

In [ ]:
from src.featurization import DrugFeaturizer, ProteinFeaturizer

drug_f = DrugFeaturizer(max_atoms=100)
prot_f = ProteinFeaturizer(max_length=1000)

# Example: featurize Aspirin
smiles = "CC(=O)Oc1ccccc1C(=O)O"
graph = drug_f.smiles_to_graph(smiles)
print(f"Drug: {smiles}")
print(f"  Atom features: {graph.node_features.shape}")
print(f"  Bonds: {graph.edge_index.shape[1]}")

# Example: featurize a protein sequence
seq = "MKTLLLTLVVVTIVCLDL"
encoding = prot_f.sequence_to_encoding(seq)
print(f"\nProtein: {seq}")
print(f"  Encoding: {encoding.shape}")
print(f"  AA composition: {prot_f.sequence_to_composition(seq)[:5]}")

## 3. Train Model

In [ ]:
with open('../configs/default.yaml') as f:
    config = yaml.safe_load(f)

print("Config loaded:")
print(f"  GNN type: {config['model']['drug_encoder']['gnn_type']}")
print(f"  Hidden dim: {config['model']['drug_encoder']['hidden_dim']}")
print(f"  Batch size: {config['training']['batch_size']}")
print(f"  Epochs: {config['training']['epochs']}")

In [ ]:
from src.model import DrugTargetPredictor

model = DrugTargetPredictor(
    drug_encoder_config=config['model']['drug_encoder'],
    protein_encoder_config=config['model']['protein_encoder'],
    fusion_type=config['model']['fusion']['type'],
    fusion_dim=config['model']['fusion']['hidden_dim']
)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model created: {total_params:,} parameters")

## 4. Evaluate Trained Model

Load the pre-trained checkpoint and evaluate on test data.

In [ ]:
from src.evaluate import calculate_metrics, generate_report
from src.train import DrugTargetDataset, collate_fn
from torch.utils.data import DataLoader

# Load checkpoint
checkpoint = torch.load('../models/checkpoints/best_model.pt', map_location='cpu', weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

# Prepare test data
train_df, val_df, test_df = dataset.split_data(df)
test_dataset = DrugTargetDataset(test_df, drug_f, prot_f)
test_loader = DataLoader(test_dataset, batch_size=256, collate_fn=collate_fn, num_workers=0)

# Evaluate
all_preds, all_targets = [], []
with torch.no_grad():
    for batch in test_loader:
        pred = model(batch['drug_data'], batch['protein'])
        all_preds.extend(pred.numpy().flatten())
        all_targets.extend(batch['target'].numpy().flatten())

metrics = calculate_metrics(np.array(all_targets), np.array(all_preds))
print(generate_report(metrics))

## 5. Make Predictions on New Drug-Target Pairs

In [ ]:
from src.predict import BindingPredictor

predictor = BindingPredictor(model)

# Predict for Imatinib vs a kinase target
result = predictor.predict(
    drug_smiles="CC1=C(C=C(C=C1)NC(=O)C2=CC=C(C=C2)CN3CCN(CC3)C)NC4=NC=CC(=N4)C5=CN=CC=C5",
    protein_sequence="MELLATGPQGASSCVPAAGQHFVVILGQGYGKVYKGEWVADANHLDFRESEQFQAFQEAELMAALGLHPHIVKIFHFYCGDLITMLVFEYCEMGSLDSYLHRKRRGALQDPYLVPTQGICKILSTILSQLKGHNLENPIDNLLDFGCRFEVQSSQSRGQSEVSEEFDEFNQACCSQSFQELWQTEEYGFGG"
)

print(f"Drug: Imatinib (Gleevec)")
print(f"Predicted pKd: {result['kd_value']:.2f}")
print(f"Binding: {result['binding_strength']}")
print(f"KD (nM): {10**(9-result['kd_value']):.0f}")

In [ ]:
# Visualize prediction distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(all_targets, all_preds, alpha=0.3, s=10)
axes[0].plot([5, 11], [5, 11], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual pKd')
axes[0].set_ylabel('Predicted pKd')
axes[0].set_title('Actual vs Predicted')
axes[0].legend()

axes[1].hist(np.array(all_targets) - np.array(all_preds), bins=50, edgecolor='black')
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_xlabel('Prediction Error')
axes[1].set_ylabel('Count')
axes[1].set_title('Error Distribution')

plt.tight_layout()
plt.show()